# 5.1 Python 直接调用 SpMV 与 GEMM

## 前置要求

具备 Python、NumPy、矩阵乘和 COO 稀疏矩阵基础；使用课程原 Ascend 910B3 环境，或本轮已验证的 A3（Ascend910_9362）/CANN 9.0，配合 `python3` 内核。

## 章节目标

- 区分“直接调用算子 API”和“加载整图模型执行”；
- 掌握 ACL 初始化、设备内存、Stream、Event 计时与资源释放；
- 分别完成 GEMM 与 COO SpMV，并按实际后端解释结果。

本章执行主线是：**ATC 准备单算子 OM → `acl.op.set_model_dir` 注册目录 → Python 准备数据 → pyACL 提交算子 → 同步 → D2H → NumPy 校验**。

这里的“直接调用”指不加载整图模型；它不表示可以跳过单算子 OM。OM 存在或 `acl` 可以导入也不等于算子已在当前设备执行通过。

<img src="images/acl_python_execution_flow.svg" width="760" style="display:block; margin-left:0;" />


## 两个算子

<table style="text-align:left; margin-left:0;">
<tr><th>算子</th><th>Python API</th><th>输入合同</th><th>CANN 9.0 实测边界</th></tr>
<tr><td>GEMM</td><td><code>acl.blas.gemm_ex</code></td><td>A[M,K]、B[K,N]、C[M,N]</td><td>设备执行路径；Kernel 类别以当前证据为准</td></tr>
<tr><td>SpMV</td><td><code>acl.op.execute_v2</code></td><td>COO indices/values、dense_shape、x</td><td>AI CPU/tf_kernel 能力验证，不能写成 AI Core 加速</td></tr>
</table>

两节都会先用 ATC 生成单算子 OM，并通过 `acl.op.set_model_dir` 注册；真正的提交入口仍是上表中的算子 API，不是 `acl.mdl.load_from_file` 一类整图推理接口。

> **证据边界：**本轮 A3 已验证：GEMM Device 路径与 SpMV AI CPU/tf_kernel Device 路径均通过 NumPy 校验。未来其它提交仍应以其新日志为证据，历史 910B3 结果不能替代当前提交证据。


In [ ]:
from pathlib import Path
import shutil
import subprocess

import acl
import numpy as np

required = {name: shutil.which(name) for name in ("atc", "npu-smi")}
missing = [name for name, path in required.items() if path is None]
if missing:
    raise RuntimeError(f"环境缺少必需命令：{', '.join(missing)}")

print("pyACL module:", Path(acl.__file__).resolve())
print("NumPy:", np.__version__)
print("Target Device ID: 0")
print("ATC path:", required["atc"])
print("ATC 实际可用性由后续 singleop 编译验证")
subprocess.run(["npu-smi", "info"], check=True)


## 章节内容

<table style="text-align:left; margin-left:0;">
<tr><th>小节</th><th>内容</th><th>入口</th></tr>
<tr><td>5.2</td><td>GEMM 单算子 OM、Event 计时和 NumPy 校验</td><td><a href="./05.02_gemm.ipynb">05.02_gemm.ipynb</a></td></tr>
<tr><td>5.3</td><td>COO SpMV 输入合同、AI CPU/tf_kernel 边界和结果校验</td><td><a href="./05.03_spmv.ipynb">05.03_spmv.ipynb</a></td></tr>
<tr><td>5.4</td><td>客观题与简单/中等/困难三档章节实践</td><td><a href="./05.04_chapter_test.ipynb">05.04_chapter_test.ipynb</a></td></tr>
</table>
